# Baseline Models on NYU Depth V2 10-Category

**Four fusion baselines using ResNet18 (0.75x width):**
1. RGB-only (single stream, 3-channel input)
2. Depth-only (single stream, 1-channel input)
3. Early fusion (RGB+Depth concatenated = 4-channel input)
4. Late fusion (two backbones, features concatenated before classifier)

All models use identical training configuration (optimizer, scheduler, grad clipping, label smoothing, dropout, epochs).
Mean class accuracy (MCA) is the primary metric.

## 1. Environment Setup & GPU Verification

In [1]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\nA100 GPU detected - optimal for training")
    elif 'V100' in gpu_name:
        print("\nV100 GPU detected - good for training (slower than A100)")
    elif 'T4' in gpu_name:
        print("\nT4 GPU detected - will be slower, consider upgrading to A100")
    else:
        print(f"\nGPU: {gpu_name}")
else:
    print("\nNO GPU DETECTED!")
    print("Enable GPU: Runtime -> Change runtime type -> Hardware accelerator: GPU")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

GPU VERIFICATION
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU Device: Tesla T4
GPU Memory: 14.56 GB

T4 GPU detected - will be slower, consider upgrading to A100



In [2]:
# Detailed GPU info
!nvidia-smi

Fri Mar 27 14:34:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [3]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

print("\nGoogle Drive mounted successfully!")
print(f"\nDrive contents:")
!ls -la /content/drive/MyDrive/ | head -20

Mounted at /content/drive

Google Drive mounted successfully!

Drive contents:
total 3117893
-rw------- 1 root root        176 Sep 21  2019 06-lab2.gdoc
-rw------- 1 root root      21621 Sep 30  2024 113-1363667-3121001@USSR24093000064918@pre-paid.png
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (1).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (2).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (3).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final.gdoc
-rw------- 1 root root        176 Jul 11  2025 2025_Gabriel_Clinger_Contractor Agreement_BASE copy.gdoc
-rw------- 1 root root      32204 Apr 18  2022 2900 On First- Welcome Home Next Steps.docx
-rw------- 1 root root       8822 Jun 24  2017 A6.docx
-rw------- 1 root root      22204 Jan 21  2023 activity (1).xlsx
-rw------- 1 root root      22161 Jan 21  2023 activity (2).xlsx
-rw------- 1 root root        176 Jan 21  2023 activity.gsheet
-rw------- 1 

## 3. Clone Repository to Local Disk (Fast I/O)

**Important:** We clone to `/content/` (local SSD) instead of Drive for 10-20x faster I/O

**Default:** Clone from GitHub (recommended - always gets latest code)

In [4]:
import os
from pathlib import Path

# Configuration
PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

os.chdir('/content')

if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"Repo already exists: {LOCAL_REPO_PATH}")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
else:
    if Path(LOCAL_REPO_PATH).exists():
        !rm -rf {LOCAL_REPO_PATH}
    print(f"Cloning from {GITHUB_REPO}...")
    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}
    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository")
    os.chdir(LOCAL_REPO_PATH)

print(f"\nWorking directory: {os.getcwd()}")
!ls -la {LOCAL_REPO_PATH}
print("\n" + "=" * 60)

REPOSITORY SETUP
Cloning from https://github.com/clingergab/Multi-Stream-Neural-Networks.git...
Cloning into '/content/Multi-Stream-Neural-Networks'...
remote: Enumerating objects: 3200, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 3200 (delta 99), reused 99 (delta 85), pack-reused 3072 (from 2)
Receiving objects: 100% (3200/3200), 117.98 MiB | 17.51 MiB/s, done.
Resolving deltas: 100% (2000/2000), done.
Encountered 49 file(s) that should have been pointers, but weren't:
	tests/augmentation_comparison.png
	tests/augmentation_test.png
	tests/balanced_augmentation_test.png
	tests/balanced_samples_comparison.png
	tests/dataset_orthogonal_loading.png
	tests/decaying_restarts_eta_min_bug.png
	tests/easing_formula_analysis.png
	tests/easing_schedulers_comparison.png
	tests/global_vs_local_comparison.png
	tests/linear_scale_comparison.png
	tests/local_vs_global_orthogonal.png
	tests/log_vs_linear_scale_effect.png
	tests/m

## 4. Install Dependencies

In [ ]:
# Install required packages
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn ray[tune] optuna kornia thop

# Verify installations
import h5py
import tqdm
import matplotlib
import seaborn
import ray
import kornia

print("All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   ray: {ray.__version__}")
print(f"   kornia: {kornia.__version__}")

## 5. Copy NYU Depth V2 Dataset to Local Disk

**Performance Note:** Local disk I/O is ~10-20x faster than Drive!

**Dataset:** NYU Depth V2 10-category preprocessed (train + test splits, RGB + Depth)

In [6]:
from pathlib import Path
import os

# Paths
DRIVE_DATASET_TAR = "/content/drive/MyDrive/datasets/nyu_depth_v2_10_traintest.tar.gz"
LOCAL_DATASET_PATH = "/dev/shm/nyu_depth_v2_10_traintest"  # Extracted location

print("=" * 60)
print("NYU Depth V2 10-CATEGORY DATASET SETUP (TRAIN + TEST)")
print("=" * 60)

# Check if already on local disk
if Path(LOCAL_DATASET_PATH).exists():
    print(f"Dataset already on local disk: {LOCAL_DATASET_PATH}")

    # Verify structure
    for split in ['train', 'test']:
        split_dir = Path(f"{LOCAL_DATASET_PATH}/{split}/rgb")
        if split_dir.exists():
            count = len(list(split_dir.glob("*.png")))
            print(f"   {split.capitalize()} samples: {count}")

# Copy and extract from Drive
elif Path(DRIVE_DATASET_TAR).exists():
    print(f"Found compressed dataset on Drive: {DRIVE_DATASET_TAR}")
    print(f"Copying compressed file to local disk...")

    # Copy compressed file with progress
    !rsync -ah --info=progress2 {DRIVE_DATASET_TAR} /dev/shm/nyu_depth_v2_10_traintest.tar.gz

    # Extract to local disk
    print(f"\nExtracting dataset to local disk...")
    !tar -xzf /dev/shm/nyu_depth_v2_10_traintest.tar.gz -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"

    # Remove tar file to save space
    !rm /dev/shm/nyu_depth_v2_10_traintest.tar.gz

    print(f"\nDataset extracted to local disk")

    # Verify extraction
    for split in ['train', 'test']:
        split_dir = Path(f"{LOCAL_DATASET_PATH}/{split}/rgb")
        if split_dir.exists():
            count = len(list(split_dir.glob("*.png")))
            print(f"   {split.capitalize()} samples: {count}")

else:
    print(f"Dataset not found on Drive!")
    print(f"   Expected location: {DRIVE_DATASET_TAR}")
    raise FileNotFoundError(f"Compressed dataset not found at {DRIVE_DATASET_TAR}")

print("\n" + "=" * 60)
print(f"Dataset ready at: {LOCAL_DATASET_PATH}")
print("=" * 60)

SUN RGB-D 15-CATEGORY DATASET SETUP (TRAIN + TEST)
Found compressed dataset on Drive: /content/drive/MyDrive/datasets/sunrgbd_19_traintest.tar.gz
Copying compressed file to local disk...
          1.54G 100%   32.27MB/s    0:00:45 (xfr#1, to-chk=0/1)

Extracting dataset to local disk...

Dataset extracted to local disk

Dataset ready at: /dev/shm/sunrgbd_19_traintest


## 6. Setup Python Path & Imports


In [ ]:
import sys
import os

# Remove cached modules
modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

project_root = '/content/Multi-Stream-Neural-Networks'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Importing core modules...")
from src.models.core.resnet import resnet18
from src.data_utils.nyu_depth_v2_dataset import get_nyu_depth_v2_dataloaders, NYUDepthV2Dataset
from src.data_utils.nyu_depth_v2_dataset import _load_norm_stats
from src.training.augmentation_config import AugmentationConfig
from src.utils.seed import set_seed

from ray import train, tune
from ray.tune.schedulers import ASHAScheduler
from ray.tune.search.optuna import OptunaSearch
from optuna.samplers import TPESampler

print("All imports successful!")

In [8]:
# Set random seed for reproducibility
from src.utils.seed import set_seed

SEED = 60  # chosen for StratifiedGroupKFold: all 10 classes in val, val=158
DETERMINISTIC = False  # False = faster, True = fully reproducible

set_seed(SEED, deterministic=DETERMINISTIC)

print(f"Seed: {SEED}, Deterministic: {DETERMINISTIC}")

Seed: 42, Deterministic: False


## 7. HPO Configuration

Ray Tune hyperparameter search settings. Each baseline runs as a separate experiment with 50 trials.

In [ ]:
import os
import subprocess
import time as _time
from pathlib import Path

from ray.tune import CLIReporter
from ray.tune import Callback as TuneCallback

# ======================== HPO SETTINGS ========================
DRIVE_STORAGE_PATH = "/content/drive/MyDrive/ray_tune_experiments"
LOCAL_STORAGE_PATH = "/content/ray_results"
EXPERIMENT_PREFIX = "baseline_hpo"

NUM_SAMPLES = 50        # Trials per baseline

# ======================== FIXED MODEL SETTINGS ========================
FIXED_CONFIG = {
    'architecture': 'resnet18',
    'num_classes': 10,
    'width_multiplier': 0.75,
    'epochs': 115,
    'warmup_epochs': 5,
    'warmup_start_factor': 0.2,
    'batch_size': 64,
    'num_workers': 1,
}

# ======================== SEARCH SPACE ========================
BASE_SEARCH_SPACE = {
    "lr": tune.loguniform(5e-5, 5e-4),
    "wd": tune.loguniform(3e-5, 6e-4),
    "eta_min": tune.loguniform(0.001, 0.015),
    "dropout_p": tune.quniform(0.25, 0.50, 0.01),
    "label_smoothing": tune.quniform(0.05, 0.15, 0.01),
    "grad_clip_norm": tune.quniform(0.8, 1.5, 0.05),
    "rgb_aug_prob": tune.quniform(0.8, 1.2, 0.01),
    "rgb_aug_mag": tune.quniform(0.8, 1.2, 0.01),
    "depth_aug_prob": tune.quniform(0.8, 1.2, 0.01),
    "depth_aug_mag": tune.quniform(0.8, 1.2, 0.01),
}

Path(DRIVE_STORAGE_PATH).mkdir(parents=True, exist_ok=True)
Path(LOCAL_STORAGE_PATH).mkdir(parents=True, exist_ok=True)

print("HPO Configuration:")
print(f"  Storage: {DRIVE_STORAGE_PATH}")
print(f"  Trials per baseline: {NUM_SAMPLES}")
print(f"  Fixed: ResNet18 0.75x, {FIXED_CONFIG['epochs']} epochs")
print(f"  Search space: {len(BASE_SEARCH_SPACE)} parameters")

## 8. Load Dataset

In [10]:
# Verify dataset structure
from pathlib import Path

print("=" * 60)
print("DATASET STRUCTURE VERIFICATION")
print("=" * 60)

dataset_root = Path(LOCAL_DATASET_PATH)

print("\nDirectory structure:")
print(f"  {dataset_root}/")
for split in ['train', 'test']:
    split_dir = dataset_root / split
    if split_dir.exists():
        print(f"    {split}/")
        for modality in ['rgb', 'depth']:
            mod_dir = split_dir / modality
            if mod_dir.exists():
                print(f"      {modality}/ - {len(list(mod_dir.glob('*.png')))} images")
        print(f"      labels.txt")

# Read class names
class_names_file = dataset_root / 'class_names.txt'
if class_names_file.exists():
    with open(class_names_file, 'r') as f:
        class_names = [line.strip() for line in f]
    print(f"\nClasses ({len(class_names)}):")
    for i, name in enumerate(class_names):
        print(f"  {i}: {name}")

print("\n" + "=" * 60)

DATASET STRUCTURE VERIFICATION

Directory structure:
  /dev/shm/sunrgbd_19_traintest/
    train/
      labels.txt
    test/
      labels.txt

Classes (19):
  0: 0: bathroom
  1: 1: bedroom
  2: 2: classroom
  3: 3: computer_room
  4: 4: conference_room
  5: 5: corridor
  6: 6: dining_area
  7: 7: dining_room
  8: 8: discussion_area
  9: 9: furniture_store
  10: 10: home_office
  11: 11: kitchen
  12: 12: lab
  13: 13: lecture_theatre
  14: 14: library
  15: 15: living_room
  16: 16: office
  17: 17: rest_space
  18: 18: study_space



In [ ]:
print("=" * 60)
print("LOADING NYU Depth V2 10-CATEGORY DATASET (TRAIN + TEST)")
print("=" * 60)

print(f"\nLoading dataset from: {LOCAL_DATASET_PATH}")

train_loader, val_loader, test_loader = get_nyu_depth_v2_dataloaders(
    data_root=LOCAL_DATASET_PATH,
    batch_size=FIXED_CONFIG['batch_size'],
    num_workers=FIXED_CONFIG['num_workers'],
    normalize=False,
)

norm_stats = _load_norm_stats(LOCAL_DATASET_PATH)

# Pre-compute scene-aware split ONCE (not per-trial)
import json as _json_split
from sklearn.model_selection import StratifiedGroupKFold
with open(os.path.join(LOCAL_DATASET_PATH, "train", "scene_groups.json")) as f:
    _scene_groups = _json_split.load(f)
with open(os.path.join(LOCAL_DATASET_PATH, "train", "labels.txt")) as f:
    _all_labels = [int(l) for l in f.read().strip().split("\n")]
_sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
_train_indices, _val_indices = next(_sgkf.split(
    list(range(len(_all_labels))), _all_labels, _scene_groups,
))
_train_indices, _val_indices = list(_train_indices), list(_val_indices)
print(f"Scene-aware split: train={len(_train_indices)}, val={len(_val_indices)}")

print(f"\nTrain samples: {len(train_loader.dataset):,}")
print(f"Test samples:  {len(test_loader.dataset):,}")
print(f"Batch size:    {FIXED_CONFIG['batch_size']}")
print(f"Norm stats loaded: {list(norm_stats.keys())}")

## 8b. Baseline HPO: ResNet18 (0.75x width)

**Four baselines** tuned independently with Ray Tune (50 trials each, ASHA early stopping):
1. **RGB-only:** Standard 3-channel input
2. **Depth-only:** Single-channel depth input
3. **Early Fusion:** 4-channel concatenated RGB+Depth
4. **Late Fusion:** Dual backbone with feature concatenation

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from collections import Counter
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from src.models.core.resnet import resnet18
import ray
from src.training.augmentation_config import AugmentationConfig
from src.data_utils.nyu_depth_v2_dataset import NYUDepthV2Dataset

# --- Dataset wrappers ---
class RGBOnlyDataset(Dataset):
    """Wraps an (rgb, depth, label) dataset to return (rgb, label)."""
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        rgb, depth, label = self.dataset[idx]
        return rgb, label

class DepthOnlyDataset(Dataset):
    """Wraps an (rgb, depth, label) dataset to return (depth, label)."""
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        rgb, depth, label = self.dataset[idx]
        return depth, label

class EarlyFusionDataset(Dataset):
    """Wraps an (rgb, depth, label) dataset to return (cat(rgb, depth), label)."""
    def __init__(self, dataset):
        self.dataset = dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        rgb, depth, label = self.dataset[idx]
        return torch.cat([rgb, depth], dim=0), label


class LateFusionResNet(nn.Module):
    def __init__(self, num_classes, width_multiplier, dropout_p):
        super().__init__()
        self.rgb_backbone = resnet18(
            num_classes=num_classes,
            width_multiplier=width_multiplier,
            device='cpu', use_amp=False
        )
        self.depth_backbone = resnet18(
            num_classes=num_classes,
            width_multiplier=width_multiplier,
            device='cpu', use_amp=False
        )
        self.depth_backbone.conv1 = nn.Conv2d(
            1, self.depth_backbone.conv1.out_channels,
            kernel_size=7, stride=2, padding=3, bias=False
        )
        nn.init.kaiming_normal_(self.depth_backbone.conv1.weight, mode='fan_out', nonlinearity='relu')

        feat_dim = self.rgb_backbone.fc.in_features
        self.rgb_backbone.fc = nn.Identity()
        self.depth_backbone.fc = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_p),
            nn.Linear(feat_dim * 2, num_classes)
        )

    def forward(self, rgb, depth):
        rgb_feat = self.rgb_backbone(rgb)
        depth_feat = self.depth_backbone(depth)
        fused = torch.cat([rgb_feat, depth_feat], dim=1)
        return self.classifier(fused)


def _build_model(fusion_type, num_classes, width_multiplier, dropout_p):
    """Build baseline model for given fusion type."""
    if fusion_type == 'rgb':
        model = resnet18(num_classes=num_classes, width_multiplier=width_multiplier,
                         device='cpu', use_amp=False)
        model.fc = nn.Sequential(nn.Dropout(p=dropout_p),
                                 nn.Linear(model.fc.in_features, model.fc.out_features))
        return model

    elif fusion_type == 'depth':
        model = resnet18(num_classes=num_classes, width_multiplier=width_multiplier,
                         device='cpu', use_amp=False)
        old_conv1 = model.conv1
        model.conv1 = nn.Conv2d(1, old_conv1.out_channels, kernel_size=7, stride=2, padding=3, bias=False)
        nn.init.kaiming_normal_(model.conv1.weight, mode='fan_out', nonlinearity='relu')
        model.fc = nn.Sequential(nn.Dropout(p=dropout_p),
                                 nn.Linear(model.fc.in_features, model.fc.out_features))
        return model

    elif fusion_type == 'early_fusion':
        model = resnet18(num_classes=num_classes, width_multiplier=width_multiplier,
                         device='cpu', use_amp=False)
        old_conv1 = model.conv1
        model.conv1 = nn.Conv2d(4, old_conv1.out_channels, kernel_size=7, stride=2, padding=3, bias=False)
        nn.init.kaiming_normal_(model.conv1.weight, mode='fan_out', nonlinearity='relu')
        model.fc = nn.Sequential(nn.Dropout(p=dropout_p),
                                 nn.Linear(model.fc.in_features, model.fc.out_features))
        return model

    elif fusion_type == 'late_fusion':
        return LateFusionResNet(num_classes, width_multiplier, dropout_p)

    else:
        raise ValueError(f"Unknown fusion_type: {fusion_type}")


def _wrap_dataset(dataset, fusion_type):
    """Wrap base dataset for the given fusion type."""
    if fusion_type == 'rgb':
        return RGBOnlyDataset(dataset)
    elif fusion_type == 'depth':
        return DepthOnlyDataset(dataset)
    elif fusion_type == 'early_fusion':
        return EarlyFusionDataset(dataset)
    elif fusion_type == 'late_fusion':
        return dataset  # returns (rgb, depth, label) directly
    else:
        raise ValueError(f"Unknown fusion_type: {fusion_type}")


def train_baseline_tune(config, fusion_type=None, data_root=None, norm_stats=None, seed=42, train_indices=None, val_indices=None, train_indices=None, val_indices=None):
    """Ray Tune trainable for baseline HPO."""
    set_seed(seed, deterministic=False)
    g = torch.Generator().manual_seed(seed)
    device = 'cuda'
    num_classes = FIXED_CONFIG['num_classes']
    epochs = FIXED_CONFIG['epochs']

    # Per-trial augmentation
    aug_config = AugmentationConfig(
        rgb_aug_prob=config.get("rgb_aug_prob", 1.0),
        rgb_aug_mag=config.get("rgb_aug_mag", 1.0),
        depth_aug_prob=config.get("depth_aug_prob", 1.0),
        depth_aug_mag=config.get("depth_aug_mag", 1.0),
    )

    # Datasets: train with aug, val without aug
    train_dataset = NYUDepthV2Dataset(data_root=data_root, split='train', normalize=False,
                                   **aug_config.to_dict())
    val_dataset = NYUDepthV2Dataset(data_root=data_root, split='train', normalize=False)
    val_dataset.split = 'val'

    # 80/20 stratified split
    all_labels = train_dataset.labels

    train_subset = torch.utils.data.Subset(
        _wrap_dataset(train_dataset, fusion_type), train_indices)
    val_subset = torch.utils.data.Subset(
        _wrap_dataset(val_dataset, fusion_type), val_indices)

    # Weighted sampler for stratified training
    subset_labels = [all_labels[i] for i in train_indices]
    label_counts = Counter(subset_labels)
    num_samples = len(subset_labels)
    class_weights = {label: num_samples / count for label, count in label_counts.items()}
    sample_weights = torch.tensor(
        [class_weights[label] for label in subset_labels], dtype=torch.float32)

    train_sampler = torch.utils.data.WeightedRandomSampler(
        weights=sample_weights, num_samples=num_samples,
        replacement=True, generator=g)

    def worker_init_fn(worker_id):
        worker_seed = seed + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)
        torch.manual_seed(worker_seed)

    train_dl = DataLoader(train_subset, batch_size=FIXED_CONFIG['batch_size'],
                          shuffle=False, sampler=train_sampler,
                          num_workers=1, prefetch_factor=2,
                          persistent_workers=True, pin_memory=True,
                          worker_init_fn=worker_init_fn)
    val_dl = DataLoader(val_subset, batch_size=FIXED_CONFIG['batch_size'],
                        shuffle=False, num_workers=1, prefetch_factor=2,
                        persistent_workers=False, pin_memory=True,
                        worker_init_fn=worker_init_fn)

    is_late_fusion = (fusion_type == 'late_fusion')

    # Build model
    model = _build_model(fusion_type, num_classes,
                         FIXED_CONFIG['width_multiplier'], config['dropout_p'])
    model.to(device)

    # Optimizer & scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'],
                                  weight_decay=config['wd'])
    warmup = LinearLR(optimizer, start_factor=FIXED_CONFIG['warmup_start_factor'],
                      end_factor=1.0, total_iters=FIXED_CONFIG['warmup_epochs'])
    cosine = CosineAnnealingLR(optimizer,
                               T_max=FIXED_CONFIG['epochs'] - FIXED_CONFIG['warmup_epochs'],
                               eta_min=config['eta_min'])
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine],
                             milestones=[FIXED_CONFIG['warmup_epochs']])

    criterion = nn.CrossEntropyLoss(label_smoothing=config['label_smoothing'])
    scaler = torch.amp.GradScaler('cuda', enabled=True)

    best_val_mca = 0.0
    best_val_acc = 0.0
    best_val_loss = float('inf')

    for epoch in range(epochs):
        # --- Train ---
        model.train()
        total_loss, correct, total = 0.0, 0, 0

        if is_late_fusion:
            for rgb, depth, labels in train_dl:
                rgb = rgb.to(device, non_blocking=True)
                depth = depth.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                optimizer.zero_grad()
                with torch.amp.autocast('cuda'):
                    logits = model(rgb, depth)
                    loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip_norm'])
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item() * labels.size(0)
                correct += logits.argmax(1).eq(labels).sum().item()
                total += labels.size(0)
        else:
            for inputs, labels in train_dl:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                optimizer.zero_grad()
                with torch.amp.autocast('cuda'):
                    logits = model(inputs)
                    loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip_norm'])
                scaler.step(optimizer)
                scaler.update()
                total_loss += loss.item() * labels.size(0)
                correct += logits.argmax(1).eq(labels).sum().item()
                total += labels.size(0)

        scheduler.step()
        train_loss = total_loss / total
        train_acc = correct / total

        # --- Validate ---
        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        v_preds, v_labels_all = [], []

        with torch.no_grad():
            if is_late_fusion:
                for rgb, depth, labels in val_dl:
                    rgb = rgb.to(device, non_blocking=True)
                    depth = depth.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True)
                    with torch.amp.autocast('cuda'):
                        logits = model(rgb, depth)
                        loss = criterion(logits, labels)
                    v_loss += loss.item() * labels.size(0)
                    preds = logits.argmax(1)
                    v_correct += preds.eq(labels).sum().item()
                    v_total += labels.size(0)
                    v_preds.append(preds.cpu())
                    v_labels_all.append(labels.cpu())
            else:
                for inputs, labels in val_dl:
                    inputs = inputs.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True)
                    with torch.amp.autocast('cuda'):
                        logits = model(inputs)
                        loss = criterion(logits, labels)
                    v_loss += loss.item() * labels.size(0)
                    preds = logits.argmax(1)
                    v_correct += preds.eq(labels).sum().item()
                    v_total += labels.size(0)
                    v_preds.append(preds.cpu())
                    v_labels_all.append(labels.cpu())

        val_loss = v_loss / v_total
        val_acc = v_correct / v_total
        v_preds = torch.cat(v_preds)
        v_labels_all = torch.cat(v_labels_all)
        per_class = [(v_preds[v_labels_all == c] == c).float().mean().item()
                     for c in range(num_classes) if (v_labels_all == c).sum() > 0]
        val_mca = sum(per_class) / len(per_class)

        # Track best
        if val_mca > best_val_mca:
            best_val_mca = val_mca
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        if val_loss < best_val_loss:
            best_val_loss = val_loss

        # Report to Ray Tune
        tune.report({
            "val_accuracy": val_acc,
            "val_loss": val_loss,
            "val_mca": val_mca,
            "best_val_mca": best_val_mca,
            "best_val_acc": best_val_acc,
            "best_val_loss": best_val_loss,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
        })


# --- Drive sync callback (copied from LINet HPO) ---
class DriveSyncCallback(TuneCallback):
    def __init__(self, local_storage_path, drive_storage_path, experiment_name,
                 sync_interval_seconds=300):
        self._local_path = os.path.join(local_storage_path, experiment_name)
        self._drive_path = os.path.join(drive_storage_path, experiment_name)
        self._sync_interval = sync_interval_seconds
        self._last_sync = 0.0

    def _sync(self, reason=""):
        if not os.path.isdir(self._local_path):
            return
        try:
            os.makedirs(self._drive_path, exist_ok=True)
            result = subprocess.run(
                ["rsync", "-a", self._local_path + "/", self._drive_path + "/"],
                capture_output=True, text=True, timeout=120)
            if result.returncode == 0:
                self._last_sync = _time.time()
                print(f"[DriveSyncCallback] synced ({reason})")
        except Exception as e:
            print(f"[DriveSyncCallback] WARNING: {e}")

    def on_trial_result(self, iteration, trials, trial, result, **info):
        if _time.time() - self._last_sync >= self._sync_interval:
            self._sync(reason=f"periodic, iter={result.get('training_iteration', '?')}")

    def on_trial_complete(self, iteration, trials, trial, **info):
        if _time.time() - self._last_sync >= 60:
            self._sync(reason="trial complete")

    def on_experiment_end(self, trials, **info):
        self._sync(reason="experiment end")


class BestTrialReporter(TuneCallback):
    def __init__(self, metric="best_val_mca", mode="max", every_n_results=10):
        self._metric = metric
        self._mode = mode
        self._every_n = every_n_results
        self._result_count = 0
        self._best_value = float('-inf') if mode == "max" else float('inf')
        self._best_config = None

    def on_trial_result(self, iteration, trials, trial, result, **info):
        self._result_count += 1
        val = result.get(self._metric)
        if val is None:
            return
        improved = (val > self._best_value) if self._mode == "max" else (val < self._best_value)
        if improved:
            self._best_value = val
            self._best_config = trial.config.copy()
        if self._result_count % self._every_n == 0 and self._best_config is not None:
            print(f"\n{'---'*20}")
            print(f"  Best {self._metric}: {self._best_value*100:.2f}% ({self._result_count} results)")
            print(f"{'---'*20}")


def run_baseline_hpo(fusion_type, data_root, norm_stats, search_space):
    """Run Ray Tune HPO for a single baseline fusion type."""
    experiment_name = f"{EXPERIMENT_PREFIX}_{fusion_type}"
    experiment_path = os.path.join(DRIVE_STORAGE_PATH, experiment_name)
    local_experiment_path = os.path.join(LOCAL_STORAGE_PATH, experiment_name)

    # Check for existing experiment to resume
    resume_existing = os.path.exists(experiment_path)
    if resume_existing:
        _exp_files = os.listdir(experiment_path) if os.path.isdir(experiment_path) else []
        if len(_exp_files) == 0:
            resume_existing = False

    if resume_existing:
        print(f"  Previous experiment found — restoring from Drive...")
        os.makedirs(local_experiment_path, exist_ok=True)
        subprocess.run(["rsync", "-a", experiment_path + "/", local_experiment_path + "/"],
                       capture_output=True, text=True)

    ray.shutdown()
    ray.init(ignore_reinit_error=True, runtime_env={
        "env_vars": {
            "CUDA_DEVICE_ORDER": "PCI_BUS_ID",
            "CUDA_VISIBLE_DEVICES": "0",
        }
    })

    trainable = tune.with_resources(
        tune.with_parameters(
            train_baseline_tune,
            fusion_type=fusion_type,
            data_root=data_root,
            norm_stats=norm_stats,
            seed=SEED,
            train_indices=_train_indices,
            val_indices=_val_indices,
        ),
        resources={"cpu": 1, "gpu": 0.1},
    )

    drive_sync_cb = DriveSyncCallback(LOCAL_STORAGE_PATH, DRIVE_STORAGE_PATH, experiment_name)

    if resume_existing:
        tuner = tune.Tuner.restore(
            path=local_experiment_path,
            trainable=trainable,
            resume_unfinished=True,
            resume_errored=True,
        )
    else:
        reporter = CLIReporter(
            parameter_columns=["lr", "wd", "dropout_p", "label_smoothing", "grad_clip_norm"],
            metric_columns={
                "training_iteration": "iter",
                "best_val_mca": "best_val_mca",
                "best_val_acc": "best_val_acc",
            },
            max_report_frequency=30,
            print_intermediate_tables=True,
        )

        optuna_search = OptunaSearch(
            metric="best_val_mca", mode="max",
            sampler=TPESampler(n_startup_trials=10),
        )

        asha_scheduler = ASHAScheduler(
            time_attr="training_iteration",
            metric="best_val_mca", mode="max",
            max_t=FIXED_CONFIG['epochs'],
            grace_period=15,
            reduction_factor=2,
        )

        tuner = tune.Tuner(
            trainable,
            param_space=search_space,
            tune_config=tune.TuneConfig(
                scheduler=asha_scheduler,
                search_alg=optuna_search,
                num_samples=NUM_SAMPLES,
                max_concurrent_trials=10,
            ),
            run_config=ray.tune.RunConfig(
                storage_path=LOCAL_STORAGE_PATH,
                name=experiment_name,
                progress_reporter=reporter,
                verbose=1,
                callbacks=[drive_sync_cb, BestTrialReporter(every_n_results=10)],
            ),
        )

    results = tuner.fit()

    best = results.get_best_result("best_val_mca", "max")
    print(f"\nBest {fusion_type} config: {best.config}")
    print(f"Best {fusion_type} val MCA: {best.metrics['best_val_mca']:.4f}")
    print(f"Best {fusion_type} val acc: {best.metrics['best_val_acc']:.4f}")

    return results


print("Trainable function and helpers defined.")
print("Fusion types: rgb, depth, early_fusion, late_fusion")

In [ ]:
# --- RGB-Only HPO ---
print('=' * 60)
print('HPO: ResNet18 0.75x width — RGB Only (50 trials)')
print('=' * 60)

# RGB-only: no depth augmentation search
rgb_search_space = {k: v for k, v in BASE_SEARCH_SPACE.items()
                    if not k.startswith('depth_aug')}
rgb_search_space['depth_aug_prob'] = 1.0
rgb_search_space['depth_aug_mag'] = 1.0

rgb_results = run_baseline_hpo('rgb', LOCAL_DATASET_PATH, norm_stats, rgb_search_space)

In [ ]:
# RGB-Only best trial summary
rgb_best = rgb_results.get_best_result("best_val_mca", "max")
print("\nRGB-Only Best Trial:")
for k, v in rgb_best.config.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.2e}" if abs(v) < 0.01 else f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")
print(f"\n  val MCA:  {rgb_best.metrics['best_val_mca']:.4f}")
print(f"  val acc:  {rgb_best.metrics['best_val_acc']:.4f}")

In [ ]:
# --- Depth-Only HPO ---
print('=' * 60)
print('HPO: ResNet18 0.75x width — Depth Only (50 trials)')
print('=' * 60)

# Depth-only: no RGB augmentation search
depth_search_space = {k: v for k, v in BASE_SEARCH_SPACE.items()
                      if not k.startswith('rgb_aug')}
depth_search_space['rgb_aug_prob'] = 1.0
depth_search_space['rgb_aug_mag'] = 1.0

depth_results = run_baseline_hpo('depth', LOCAL_DATASET_PATH, norm_stats, depth_search_space)

In [ ]:
# Depth-Only best trial summary
depth_best = depth_results.get_best_result("best_val_mca", "max")
print("\nDepth-Only Best Trial:")
for k, v in depth_best.config.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.2e}" if abs(v) < 0.01 else f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")
print(f"\n  val MCA:  {depth_best.metrics['best_val_mca']:.4f}")
print(f"  val acc:  {depth_best.metrics['best_val_acc']:.4f}")

In [ ]:
# --- Early Fusion HPO ---
print('=' * 60)
print('HPO: ResNet18 0.75x width — Early Fusion (50 trials)')
print('=' * 60)

# Early fusion uses all augmentation params
ef_results = run_baseline_hpo('early_fusion', LOCAL_DATASET_PATH, norm_stats, BASE_SEARCH_SPACE.copy())

In [ ]:
# Early Fusion best trial summary
ef_best = ef_results.get_best_result("best_val_mca", "max")
print("\nEarly Fusion Best Trial:")
for k, v in ef_best.config.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.2e}" if abs(v) < 0.01 else f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")
print(f"\n  val MCA:  {ef_best.metrics['best_val_mca']:.4f}")
print(f"  val acc:  {ef_best.metrics['best_val_acc']:.4f}")

In [ ]:
# --- Late Fusion HPO ---
print('=' * 60)
print('HPO: ResNet18 0.75x width — Late Fusion (50 trials)')
print('=' * 60)

# Late fusion uses all augmentation params
lf_results = run_baseline_hpo('late_fusion', LOCAL_DATASET_PATH, norm_stats, BASE_SEARCH_SPACE.copy())

In [ ]:
# Late Fusion best trial summary
lf_best = lf_results.get_best_result("best_val_mca", "max")
print("\nLate Fusion Best Trial:")
for k, v in lf_best.config.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.2e}" if abs(v) < 0.01 else f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")
print(f"\n  val MCA:  {lf_best.metrics['best_val_mca']:.4f}")
print(f"  val acc:  {lf_best.metrics['best_val_acc']:.4f}")

## 9. HPO Results Comparison

In [ ]:
import matplotlib.pyplot as plt

# Collect best results
all_results = {
    'RGB-Only': rgb_results,
    'Depth-Only': depth_results,
    'Early Fusion': ef_results,
    'Late Fusion': lf_results,
}

# --- Best trial MCA bar chart ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(all_results.keys())
best_mcas = []
best_accs = []

for name, results in all_results.items():
    best = results.get_best_result("best_val_mca", "max")
    best_mcas.append(best.metrics['best_val_mca'] * 100)
    best_accs.append(best.metrics['best_val_acc'] * 100)

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

axes[0].bar(names, best_mcas, color=colors)
axes[0].set_ylabel('Val MCA (%)')
axes[0].set_title('Best Val Mean Class Accuracy')
for i, v in enumerate(best_mcas):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

axes[1].bar(names, best_accs, color=colors)
axes[1].set_ylabel('Val Accuracy (%)')
axes[1].set_title('Best Val Accuracy')
for i, v in enumerate(best_accs):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Baseline HPO Summary

In [ ]:
# --- Summary table ---
print('=' * 80)
print('BASELINE HPO RESULTS — ResNet18 (0.75x width) on NYU Depth V2 10-Category')
print('=' * 80)

print(f"{'Model':<18} {'Val MCA':>10} {'Val Acc':>10} {'Best LR':>12} {'Best WD':>12} {'Dropout':>10}")
print('-' * 80)

for name, results in all_results.items():
    best = results.get_best_result("best_val_mca", "max")
    m = best.metrics
    c = best.config
    print(f"{name:<18} {m['best_val_mca']*100:>9.2f}% {m['best_val_acc']*100:>9.2f}% "
          f"{c['lr']:>12.2e} {c['wd']:>12.2e} {c['dropout_p']:>9.2f}")

print('=' * 80)
print()

# Print full best configs
for name, results in all_results.items():
    best = results.get_best_result("best_val_mca", "max")
    print(f"\n--- {name} best config ---")
    for k, v in sorted(best.config.items()):
        if isinstance(v, float):
            print(f"  {k}: {v:.6f}" if abs(v) >= 0.01 else f"  {k}: {v:.2e}")
        else:
            print(f"  {k}: {v}")